In [138]:
import sqlite3
from pathlib import Path
import logging

In [139]:
# BASE verwijst naar de huidige map waarin het script wordt uitgevoerd
BASE = Path.cwd()

# Pad naar de centrale SDM-database
SDM = BASE / 'SDM.db'

# Pad naar het tekstbestand met het databaseschema
SCHEMA = BASE / 'BikeToDrive_RIM - SDM.txt'

# Overzicht van alle bron-databases
# De sleutel is een korte naam, de waarde is het pad naar het .db-bestand
SOURCES = {
    'accessoire_inkoop': BASE / 'BikeToDrive_4_Accessoire_Inkoop.db',
    'accessoireverkoop': BASE / 'BikeToDrive_1_Accessoireverkoop.db',
    'onderhoud': BASE / 'BikeToDrive_3_Onderhoud.db',
    'fiets_inkoop': BASE / 'BikeToDrive_5_Fiets_Inkoop.db',
    'fietsverkoop': BASE / 'BikeToDrive_2_Fietsverkoop.db',
}

# Opbouw per tuple:
# (bron_database, bron_tabel, doel_tabel_in_SDM)
MAPS = [
    ('accessoire_inkoop', 'Leverancier', 'Accessoire_Inkoop_Leverancier'),
    ('accessoire_inkoop', 'Accessoire', 'Accessoire_Inkoop_Accessoire'),
    ('accessoire_inkoop', 'Accessoire_Inkoop', 'Accessoire_Inkoop'),

    ('accessoireverkoop', 'Filiaal', 'Accessoireverkoop_Filiaal'),
    ('accessoireverkoop', 'Leverancier', 'Accessoireverkoop_Leverancier'),
    ('accessoireverkoop', 'Klant', 'Accessoireverkoop_Klant'),
    ('accessoireverkoop', 'Monteur', 'Accessoireverkoop_Monteur'),
    ('accessoireverkoop', 'Accessoire', 'Accessoireverkoop_Accessoire'),
    ('accessoireverkoop', 'Accessoire_Verkoop', 'Accessoireverkoop_Accessoire_Verkoop'),

    ('onderhoud', 'Fabrikant', 'Onderhoud_Fabrikant'),
    ('onderhoud', 'Filiaal', 'Onderhoud_Filiaal'),
    ('onderhoud', 'Fiets', 'Onderhoud_Fiets'),
    ('onderhoud', 'Monteur', 'Onderhoud_Monteur'),
    ('onderhoud', 'Onderhoud', 'Onderhoud'),

    ('fiets_inkoop', 'Fabrikant', 'Fiets_Inkoop_Fabrikant'),
    ('fiets_inkoop', 'Fiets', 'Fiets_Inkoop_Fiets'),
    ('fiets_inkoop', 'Fiets_Inkoop', 'Fiets_Inkoop'),

    ('fietsverkoop', 'Filiaal', 'Fietsverkoop_Filiaal'),
    ('fietsverkoop', 'Klant', 'Fietsverkoop_Klant'),
    ('fietsverkoop', 'Fabrikant', 'Fietsverkoop_Fabrikant'),
    ('fietsverkoop', 'Monteur', 'Fietsverkoop_Monteur'),
    ('fietsverkoop', 'Fiets', 'Fietsverkoop_Fiets'),
    ('fietsverkoop', 'Fiets_Verkoop', 'Fietsverkoop_Fiets_Verkoop'),
]

In [140]:
# Zet een naam tussen dubbele aanhalingstekens
def q(name):
    return f'"{name}"'

In [141]:
# Verbind met SDM
sdm = sqlite3.connect(SDM)

# Zet foreign keys aan
sdm.execute('PRAGMA foreign_keys = ON')

# Verbind met bron-databases
sources = {k: sqlite3.connect(v) for k, v in SOURCES.items()}

In [142]:
# full refresh
sdm.execute('PRAGMA foreign_keys = OFF')

# Loop door alle mappings heen
for _, _, target in MAPS:
    # Verwijder alle gegevens uit de doeltabel
    sdm.execute(f'DELETE FROM {q(target)}')

sdm.commit()

sdm.execute('PRAGMA foreign_keys = ON')

print("SDM volledig geleegd (Full Refresh)")

SDM volledig geleegd (Full Refresh)


In [143]:
# String-built SQL
# Loop door alle mappings
for db, source_table, target_table in MAPS:
    # Laat zien welke tabel wordt geladen
    print(f'Loading {db}.{source_table} -> {target_table}')

    # Haal alle rijen op uit de brontabel
    rows = sources[db].execute(
        f'SELECT * FROM {q(source_table)}'
    ).fetchall()

    # Ga verder als er geen data is
    if not rows:
        continue

    # Maak placeholders voor de INSERT-query
    placeholders = ', '.join(['?'] * len(rows[0]))

    # Voeg alle rijen toe aan de doeltabel
    sdm.executemany(
        f'INSERT INTO {q(target_table)} VALUES ({placeholders})',
        rows
    )

sdm.commit()

print("Load compleet")

Loading accessoire_inkoop.Leverancier -> Accessoire_Inkoop_Leverancier
Loading accessoire_inkoop.Accessoire -> Accessoire_Inkoop_Accessoire
Loading accessoire_inkoop.Accessoire_Inkoop -> Accessoire_Inkoop
Loading accessoireverkoop.Filiaal -> Accessoireverkoop_Filiaal
Loading accessoireverkoop.Leverancier -> Accessoireverkoop_Leverancier
Loading accessoireverkoop.Klant -> Accessoireverkoop_Klant
Loading accessoireverkoop.Monteur -> Accessoireverkoop_Monteur
Loading accessoireverkoop.Accessoire -> Accessoireverkoop_Accessoire
Loading accessoireverkoop.Accessoire_Verkoop -> Accessoireverkoop_Accessoire_Verkoop
Loading onderhoud.Fabrikant -> Onderhoud_Fabrikant
Loading onderhoud.Filiaal -> Onderhoud_Filiaal
Loading onderhoud.Fiets -> Onderhoud_Fiets
Loading onderhoud.Monteur -> Onderhoud_Monteur
Loading onderhoud.Onderhoud -> Onderhoud
Loading fiets_inkoop.Fabrikant -> Fiets_Inkoop_Fabrikant
Loading fiets_inkoop.Fiets -> Fiets_Inkoop_Fiets
Loading fiets_inkoop.Fiets_Inkoop -> Fiets_Inkoop


In [144]:
# Haal alle gewone tabellen op
tables = [
    r[0] for r in sdm.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
    )
]

# Loop door alle tabellen
for t in tables:
    # Tel het aantal rijen in de tabel
    count = sdm.execute(f'SELECT COUNT(*) FROM {q(t)}').fetchone()[0]

    # Toon tabelnaam en aantal rijen
    print(f'{t}: {count}')

Accessoire_Inkoop_Leverancier: 5
Accessoire_Inkoop_Accessoire: 13
Accessoire_Inkoop: 50
Accessoireverkoop_Filiaal: 4
Accessoireverkoop_Klant: 20
Accessoireverkoop_Leverancier: 5
Accessoireverkoop_Accessoire: 10
Accessoireverkoop_Monteur: 10
Accessoireverkoop_Accessoire_Verkoop: 100
Onderhoud_Fabrikant: 11
Onderhoud_Fiets: 30
Onderhoud_Filiaal: 5
Onderhoud_Monteur: 15
Onderhoud: 50
Fiets_Inkoop_Fabrikant: 10
Fiets_Inkoop_Fiets: 75
Fiets_Inkoop: 100
Fietsverkoop_Filiaal: 4
Fietsverkoop_Klant: 25
Fietsverkoop_Monteur: 10
Fietsverkoop_Fabrikant: 10
Fietsverkoop_Fiets: 75
Fietsverkoop_Fiets_Verkoop: 150


In [145]:
for c in sources.values():
    c.close()

sdm.close()